In [ ]:
from pymodulon.core import IcaData
from pymodulon.plotting import *
from pymodulon.enrichment import *

from os import path
import pandas as pd
import re
from Bio.KEGG import REST
from tqdm.notebook import tqdm
from collections import defaultdict

In [ ]:
ica_data_dir = '../data/ica_runs_prot/ica_runs/100/'

## Load data and create ICA object

#### Metadata

In [ ]:
df_metadata = pd.read_csv('../data/processed_data/metadata.tsv',index_col=0, sep='\t')
display(
    df_metadata.head(),
    df_metadata.shape
)

In [ ]:
print(df_metadata.project.notnull().all())
print(df_metadata.condition.notnull().all())

#### ICA Data

In [ ]:
A=pd.read_csv(path.join(ica_data_dir,'A.csv'),index_col=0)
M=pd.read_csv(path.join(ica_data_dir,'M.csv'),index_col=0)
X=pd.read_csv('../data/processed_data/log_normalizedCounts_norm.csv',index_col=0)
set(X.columns)-set(A.columns)
A[X.columns].to_csv(path.join(ica_data_dir,'A.csv'))

In [ ]:
X_log_tpm=pd.read_csv('../data/processed_data/log_normalizedCounts.csv',index_col=0)
set(X.columns)-set(X_log_tpm.columns)
X_log_tpm = X_log_tpm[X.columns]

In [ ]:
for matrix in M, X, X_log_tpm:
    for index in matrix.index: # remove 'gene-' from each gene 
        matrix.rename(index={index:index.strip('gene-')},inplace=True)

##### TRN

In [ ]:
trn = pd.read_csv('../data/external/trn.csv', index_col=0).dropna()
trn = trn[["reg","gene_name", "gene_id", "effect"]]
trn = trn.rename({"reg":"regulator"}, axis=1)


count = 0
genes_in_regulons = trn["gene_id"].unique()
for gene in X.index:
    if gene in genes_in_regulons:
        count+=1
print("Number of genes with annotated regulators: ",count)
print("Number of unique regulators:", len(trn.regulator.unique()))
#filter trn to only include genes which were aligned to
display(trn.head(), trn.shape)

In [ ]:
print(trn.regulator.notnull().all())
print(trn.gene_id.notnull().all())

In [ ]:
chr_info = pd.read_csv('../data/processed_data/chromosome_info.csv', index_col=0)

In [ ]:
#M matrix renamed as S matrix

ica_data = IcaData(M = M,
                   A = A,
                   X = X,
                   log_tpm = X_log_tpm,
                   gene_table = '../data/processed_data/gene_info.csv',
                   sample_table = '../data/processed_data/metadata.tsv',
                   trn = trn[trn.gene_id.isin(X.index.to_list())],
                   chrom=chr_info
                  )

In [ ]:
# set value for optimal threshold after it has been calculated the first time
ica_data.recompute_thresholds(500)

In [ ]:
from pymodulon.util import explained_variance
print("ICA Total Explained Variance:", explained_variance(ica_data))

# add individual explained variance for each iModulon

for k in ica_data.imodulon_table.index:
    ica_data.imodulon_table.loc[k, 'exp_var'] = explained_variance(
        ica_data, imodulons=k)

In [ ]:
Gene_Presence_Matrix = pd.DataFrame(index = ica_data.M.index, columns = ica_data.imodulon_names).fillna(False)

for im in ica_data.imodulon_names:
    genes = ica_data.view_imodulon(im).index
    Gene_Presence_Matrix.loc[genes, im] = True

count = Gene_Presence_Matrix.sum(axis=1)[Gene_Presence_Matrix.sum(axis=1) > 0].shape[0]
multi_imod_count = Gene_Presence_Matrix.sum(axis=1)[Gene_Presence_Matrix.sum(axis=1) > 1].shape[0]

print("There are ", count, " unique genes captured within the imodulons.")
print("There are ", multi_imod_count, " unique genes captured within multiple imodulons.")

# Adjust Thresholds

In [ ]:
# for iModulons with no genes based on kurtosis metric, adjust to include top 1% of genes for enrichment purposes
for im in ica_data.imodulon_table.index:
    if len(ica_data.view_imodulon(im)) == 0:
        lenient_threshold = ica_data.M[im].abs().quantile(0.99)
        ica_data.thresholds[im] =  lenient_threshold
        ica_data.change_threshold(im, lenient_threshold)

# for iModulons with overly strict threshold or thresholding based on manual curation, adjust to include top 1% of genes for enrichment
ims_to_adjust = [73, 72, 69, 14]

for im in ims_to_adjust:
    lenient_threshold = ica_data.M[im].abs().quantile(0.99)
    ica_data.thresholds[im] =  lenient_threshold
    ica_data.change_threshold(im, lenient_threshold)

# Load Enrichments From TRN, KEGG, and GO

In [ ]:
TRN_ENRICHMENTS = path.join('..','data', 'processed_data', 'trn_enrichments.csv')
TRN_ENRICHMENTS_SUPP = path.join('..','data', 'processed_data', 'trn_enrichments_supp.csv')
KEGG_ENRICHMENTS_PATHWAYS = path.join('..','data', 'processed_data', 'kegg_pathway_enrichments.csv')
KEGG_ENRICHMENTS_MODULES = path.join('..','data', 'processed_data', 'kegg_module_enrichments.csv')
GO_ENRICHMENTS = path.join('..','data', 'processed_data', 'go_enrichment.csv')

In [ ]:
trn_enrichments = pd.read_csv(TRN_ENRICHMENTS, index_col=0, low_memory=False)
trn_enrichments_supp = pd.read_csv(TRN_ENRICHMENTS_SUPP, index_col=0, low_memory=False)
kegg_pathway_enrichments = pd.read_csv(KEGG_ENRICHMENTS_PATHWAYS, index_col=0, low_memory=False)
kegg_module_enrichments = pd.read_csv(KEGG_ENRICHMENTS_MODULES, index_col=0, low_memory=False)
go_enrichments = pd.read_csv(GO_ENRICHMENTS, index_col=0, low_memory=False)

## Naming iModulons Based on Single Gene Status

In [ ]:
sg_imods = ica_data.find_single_gene_imodulons(save=True)
print(sg_imods)

In [ ]:
for iM in sg_imods:
    new_thresh = ica_data.M[iM].max() * .9 # threshold is 90% of max of top value for SG iMods
    ica_data.thresholds[iM] =  new_thresh # chosen to mark a point at which the highest weighted gene dominates imodulon
    top_gene = ica_data.view_imodulon(iM).gene_name[0]
    ica_data.change_threshold(iM, new_thresh)
    # ica_data.rename_imodulons({iM:'single_gene_'+str(top_gene)})

ica_data.imodulon_table.single_gene.fillna(False, inplace=True)

    
# add imodulon size for each imodulon
for i in ica_data.imodulon_table.index:
    ica_data.imodulon_table.at[i, "imodulon_size"] = len(ica_data.view_imodulon(i))

In [ ]:
M_filtered = ica_data.M.copy()
M_filtered = (M_filtered * (M_filtered.abs() > ica_data.thresholds.values()))

# calculate explained variance while excluding sub-threshold genes
baseline = ica_data.X

# Base error (variance of original centered data)
base_err = np.linalg.norm(baseline) ** 2

# Reconstruction: M @ A
MA = M_filtered.values @ ica_data.A.values

# Reconstruction error
sa_err = np.linalg.norm(MA - baseline) ** 2

# Variance explained
var_explained = np.clip(1 - sa_err / base_err, 0, 1)

print("Variance explained by genes in iMs:", var_explained)

## Naming iModulons Based on Associated Regulators

In [ ]:
# go through each iM and name based on regulatory enrichments
for iM in tqdm(ica_data.imodulon_table.index):
    if 'single_gene' in str(iM):
        continue

    enrichment_rule = 1

    # first, check expanded TRN for enrichments for iModulons
    im_enrichments = trn_enrichments[trn_enrichments.imodulon == (iM)]
    im_enrichments = im_enrichments.sort_values('qvalue')

    # if no enrichments for prior method, check single regulators for increased statistical power
    if len(im_enrichments) == 0:
        try:
            im_enrichments = ica_data.compute_trn_enrichment(imodulons=[iM], max_regs=1)
            enrichment_rule = 2
        except Exception as e:
            # print(f"Error during enrichment for {iM} with max_regs=1: {e}")
            im_enrichments = pd.DataFrame()  # fallback to empty DataFrame
    
    if len(im_enrichments) == 0:
        im_enrichments = trn_enrichments_supp[trn_enrichments_supp.imodulon == (iM)]
        im_enrichments = im_enrichments.sort_values('qvalue')
        enrichment_rule = 3
    

    


    if len(im_enrichments) > 0:
        ica_data.imodulon_table.loc[iM, 'regulator'] = im_enrichments.iloc[0, 1]
        ica_data.imodulon_table.loc[iM, 'pvalue'] = float(im_enrichments.iloc[0, 2])
        ica_data.imodulon_table.loc[iM, 'qvalue'] = im_enrichments.iloc[0, 3]
        ica_data.imodulon_table.loc[iM, 'precision'] = im_enrichments.iloc[0, 4]
        ica_data.imodulon_table.loc[iM, 'recall'] = im_enrichments.iloc[0, 5]
        ica_data.imodulon_table.loc[iM, 'f1score'] = im_enrichments.iloc[0, 6]
        ica_data.imodulon_table.loc[iM, 'TP'] = im_enrichments.iloc[0, 7]
        ica_data.imodulon_table.loc[iM, 'regulon_size'] = im_enrichments.iloc[0, 8]
        ica_data.imodulon_table.loc[iM, 'n_regs'] = int(im_enrichments.iloc[0, 10])
        ica_data.imodulon_table.loc[iM, 'enrichment_rule'] = enrichment_rule

ica_data.imodulon_table = ica_data.imodulon_table.astype({
    'pvalue': 'float64',
    'qvalue': 'float64',
    'precision': 'float64',
    'recall': 'float64',
    'f1score': 'float64',
    'TP': 'Int64',
    'regulon_size': 'Int64',
    'n_regs': 'Int64',
    'enrichment_rule':'Int64'
})

# reg_counts_total = ica_data.imodulon_table.regulator.value_counts()
# reg_counts_running = defaultdict(int)
# for iM in ica_data.imodulon_table.sort_values('exp_var', ascending=False).index:
#     if not pd.isna(ica_data.imodulon_table.loc[iM, 'regulator']):
#         reg = ica_data.imodulon_table.loc[iM, 'regulator']
#         precision = ica_data.imodulon_table.loc[iM, 'precision']
#         if precision > .4:
#             if reg_counts_total[reg] > 1:
#                 reg_counts_running[reg] += 1
#                 ica_data.rename_imodulons({iM:reg+'-'+str(reg_counts_running[reg])})
#             else:
#                 ica_data.rename_imodulons({iM:reg})


In [ ]:
ica_data.imodulon_table.reset_index().to_csv('../data/interim/imodulon_table.csv')

In [ ]:
len([x for x in ica_data.imodulon_table.regulator if type(x) == type('') and 'single' not in x])

In [ ]:
compare_imodulons_vs_regulons(ica_data,
                              size_column='imodulon_size',
                              cat_column = 'regulator',
                              scale=3);

## Naming iModulons Based on GO Enrichments

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(ica_data.imodulon_table.sort_values('exp_var', ascending=False))

In [ ]:
for iM in range(len(ica_data.imodulon_table.index)):
    im_name = ica_data.imodulon_table.index[iM]
    temp_kegg = kegg_pathway_enrichments[kegg_pathway_enrichments.imodulon == iM]
    temp_go = go_enrichments[go_enrichments.imodulon == (iM)]
    temp_go = temp_go.sort_values('qvalue')
    display(im_name, temp_go, temp_kegg)

# Visualize iMs

In [ ]:
im  = 73

plot_activities(ica_data, im)
plot_gene_weights(ica_data, im)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(ica_data.view_imodulon(im).sort_values('gene_weight'))

In [ ]:
plot_activities(ica_data, im, projects=['uw_stress_response'])

# Name iMs from iM Table

In [ ]:
iM_table = pd.read_csv('../data/processed_data/iModulon_table_annotated.csv', index_col=0)

for im, row in iM_table.iterrows():
    ica_data.rename_imodulons({im:row['name']})

ica_data.imodulon_table = iM_table.set_index('name')

# Activity Clustering

In [ ]:
sns.clustermap(ica_data.A.T.corr(method='spearman'), figsize=(30,30), cmap='coolwarm', center=0)

In [ ]:
import matplotlib.cm as cm
'''
The samples list define all sample names for which PCA is done, the labels plot labels the sample type for 
the scatter plot
'''
A = ica_data.A.copy()
A.index = [str(x) for x in A.index]

# Perform PCA
pca = PCA(n_components=3)
principal_components = pca.fit_transform(A.T)

# Create a new DataFrame with the principal components
df_pca = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2', 'PC3'], index = A.columns)

labels = ica_data.sample_table.project.values

# Choose a colormap
cmap = cm.get_cmap('tab20', 40)  # 10 discrete colors
rgb_colors = [cmap(i) for i in range(cmap.N)]
color_map = dict(zip(ica_data.sample_table.project.unique(), rgb_colors))

# Set up the scatter plots
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plot each label with a different color and marker
for index, label in zip(ica_data.sample_table.index, labels):
    color = color_map.get(label)
    
    
    ax.scatter(df_pca.loc[index, 'PC1'], df_pca.loc[index, 'PC2'], label=label, alpha=0.7, s=200, color=color)
# ax.scatter(df_pca.iloc[:, 0], df_pca.iloc[:, 1], alpha=0.7, s=200)

ax.set_xlabel(f'PC{1} - Explained Variance: {pca.explained_variance_ratio_[0]:.2%}',fontsize=20)
ax.set_ylabel(f'PC{2} - Explained Variance: {pca.explained_variance_ratio_[1]:.2%}',fontsize=20)
ax.legend()


##Add in arrows for biplot
loadings = pd.DataFrame(index=A.index)
loadings['PC1'] = pca.components_[0, :]
loadings['PC2'] = pca.components_[1, :]

arrows_to_make = list(loadings['PC1'].abs().sort_values(ascending = False).index[:3]) + list(loadings['PC2'].abs().sort_values(ascending = False).index[:3])
arrows_to_make = sorted(set(arrows_to_make))


# arrow length variable extends arrows to help in interpreting the plot
# text_scale determines how far text labels are from arrow tips (1 = on top of them)
arrow_length = 50
text_scale = 1.1

# the place the arrows start is up to you, so you can move it out of the way
start1 = 0
start2 = 0

# loop through each gene and add its arrow
for g in arrows_to_make:
    # directions in PC1 and 2
    g_weight_pc1 = loadings.loc[g, 'PC1']
    g_weight_pc2 = loadings.loc[g, 'PC2']
    
    # lengthen arrow by a constant factor
    g_weight_pc1 = g_weight_pc1 * arrow_length
    g_weight_pc2 = g_weight_pc2 * arrow_length
    
    # add the arrow
    ax.arrow(start1, start2, g_weight_pc1, g_weight_pc2, head_width = 1,color='black')

    ax.text(start1 + (text_scale * g_weight_pc1)+1,
            start2 + (text_scale * g_weight_pc2), 
            g,size=10)


from matplotlib.lines import Line2D

# Create custom legend handles for each project
unique_labels = ica_data.sample_table.project.unique()
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label=label,
           markerfacecolor=color_map[label], markersize=10)
    for label in unique_labels
]

ax.legend(handles=legend_elements, title='Project', fontsize=12, title_fontsize=13, loc='center left', bbox_to_anchor=(1, 0.5))

# Show the plots
plt.tight_layout()

# Save to JSON

In [ ]:
from pymodulon.io import *
save_to_json(ica_data,'../modulome_sc.json')